# Notebook 01: Hyperspectral Water-Quality Data Exploration

## Objective
Welcome to the first notebook of the **Hyperspectral Water-Quality Monitoring** project!
In this notebook, we will:
1. Load the raw hyperspectral reflectance dataset.
2. Inspect the dataset dimensions, sample count, and feature columns.
3. Check for missing values (`NaN`) and duplicate samples.
4. Analyze the statistical distribution of our target variable: **Turbidity ($NTU$)**.
5. Plot and analyze **Hyperspectral Reflectance Signatures** across wavelengths ($400nm - 900nm$).
6. Understand what the physical spectral graphs mean for water quality.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path to import project modules
sys.path.append(os.path.abspath("../src"))
from data_loader import load_dataset, BAND_COLS, TARGET_COL, WAVELENGTHS

# Set aesthetic plot style
sns.set_theme(style="whitegrid")
print("✓ Libraries successfully imported!")

--- 
## Step 1: Load Dataset
We load our dataset using the modular `data_loader.py` utility.

In [ ]:
raw_data_path = os.path.join("..", "data", "raw", "water_quality_hyperspectral_data.csv")
df, band_cols, target_col = load_dataset(raw_data_path)

print(f"Dataset Shape: {df.shape}")
print(f"Target Column (y): {target_col}")
print(f"Number of Hyperspectral Bands (X): {len(band_cols)}")

df.head()

--- 
## Step 2: Data Cleaning & Integrity Check
Before training any ML model, we MUST verify:
- Are there missing values (`NaN`)?
- Are there duplicate observations?

In [ ]:
# Check missing values
missing_values = df.isnull().sum().sum()
print(f"Total Missing Values: {missing_values}")

# Check duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Total Duplicate Rows: {duplicate_rows}")

--- 
## Step 3: Target Parameter Distribution (Turbidity NTU)
Let's inspect the target variable's summary statistics and histogram.

In [ ]:
df[target_col].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df[target_col], kde=True, color='#1f77b4', bins=30, edgecolor='black')
plt.axvline(df[target_col].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df[target_col].mean():.2f} NTU")
plt.axvline(df[target_col].median(), color='green', linestyle='-', linewidth=2, label=f"Median: {df[target_col].median():.2f} NTU")

plt.title("Target Distribution: Turbidity (NTU)", fontsize=14, fontweight='bold')
plt.xlabel("Turbidity (NTU)", fontsize=12)
plt.ylabel("Sample Count", fontsize=12)
plt.legend()
plt.show()

--- 
## Step 4: Spectral Signature Analysis
Hyperspectral sensors record surface reflectance $R_{rs}(\lambda)$ across continuous wavelength bands ($400nm - 900nm$).

### Physical Spectral Behavior:
- **Higher Turbidity**: Causes increased particle scattering, boosting overall spectral reflectance (especially in the Red and Near Infrared regions $650nm - 850nm$).
- **Lower Turbidity (Clear Water)**: Pure water absorbs light strongly in the NIR region, resulting in near-zero reflectance above $750nm$.

Let's plot the average spectral curves for different turbidity levels!

In [ ]:
# Divide turbidity into 4 quartile categories for spectral comparison
df['turbidity_group'] = pd.qcut(df[target_col], q=4, labels=['Clear Water (Q1)', 'Moderate (Q2)', 'Turbid (Q3)', 'Highly Turbid (Q4)'])

plt.figure(figsize=(12, 6))
palette = sns.color_palette("viridis", 4)

for idx, (group_name, group_df) in enumerate(df.groupby('turbidity_group', observed=False)):
    mean_spectrum = group_df[band_cols].mean(axis=0).values
    plt.plot(WAVELENGTHS, mean_spectrum, label=f"{group_name} (Mean: {group_df[target_col].mean():.1f} NTU)", linewidth=2.5, color=palette[idx])

plt.title("Hyperspectral Reflectance Signatures by Turbidity Group", fontsize=14, fontweight='bold')
plt.xlabel("Wavelength (nm)", fontsize=12)
plt.ylabel("Remote Sensing Reflectance R_rs (sr⁻¹)", fontsize=12)
plt.legend()
plt.show()

--- 
## Conclusion & Next Steps
1. The raw hyperspectral dataset has **500 samples** and **51 feature bands** ($400nm - 900nm$).
2. There are **no missing values** or **duplicate rows**.
3. Higher turbidity clearly correlates with elevated reflectance curves in the $600nm - 850nm$ spectral range.
4. In **Phase 2**, we will build our preprocessing pipeline and train baseline ML models!